# Predição das embeddings por patch

Este notebook carrega o CSV processado com embeddings e treina modelos de regressão multivariada para prever os 768 valores da embedding usando as demais colunas como features.

Importante:
- o split é feito por paciente (`patient_id`), para que todas as 197 linhas de uma imagem fiquem no mesmo conjunto;
- o alvo é o vetor de 768 embeddings (`emb_0` ... `emb_767`);
- os features são todas as colunas restantes, incluindo as variáveis clínicas e as coordenadas `x_pos` e `y_pos`.

In [1]:
# Imports
import re
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

# Modelos
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [2]:
# Carregamento do dataset
DATA_PATH = "dataset_breast/processed_dataset_embeddings.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"Pacientes únicos: {df['patient_id'].nunique()}")
print(f"Linhas por paciente: min={df.groupby('patient_id').size().min()}, max={df.groupby('patient_id').size().max()}")

Dataset carregado: 50432 linhas x 902 colunas
Pacientes únicos: 256
Linhas por paciente: min=197, max=197


In [3]:
# Definição de alvo e features
embedding_cols = [col for col in df.columns if col.startswith("emb_")]
feature_cols = [col for col in df.columns if col not in embedding_cols]
feature_cols = [col for col in feature_cols if col != "patient_id"]

# O LightGBM rejeita nomes de colunas com caracteres especiais como espaços, barras, parênteses, apóstrofos etc.
# Normalizamos todos os nomes das features antes do treino sem alterar os dados originais.
feature_cols_clean = [re.sub(r"[^A-Za-z0-9_]+", "_", str(col)).strip("_") for col in feature_cols]
feature_rename_map = dict(zip(feature_cols, feature_cols_clean))

# Se você quiser incluir patient_id como feature, remova a linha acima.
# Aqui mantemos o identificador fora do modelo para evitar overfitting por paciente.

X = df[feature_cols].copy().rename(columns=feature_rename_map)
y = df[embedding_cols].copy().astype(float)

print(y.head(2).to_string(index=False))

print(f"Features: {len(feature_cols_clean)}")
print(list(X.columns[:10]))
print(f"Target: {len(embedding_cols)} embeddings")

  emb_0     emb_1     emb_2     emb_3     emb_4     emb_5     emb_6     emb_7     emb_8     emb_9    emb_10    emb_11   emb_12    emb_13    emb_14    emb_15    emb_16   emb_17    emb_18   emb_19    emb_20   emb_21   emb_22    emb_23    emb_24   emb_25    emb_26    emb_27   emb_28   emb_29    emb_30    emb_31    emb_32    emb_33   emb_34    emb_35    emb_36   emb_37   emb_38    emb_39   emb_40    emb_41    emb_42   emb_43   emb_44    emb_45    emb_46   emb_47    emb_48    emb_49    emb_50    emb_51    emb_52    emb_53    emb_54    emb_55    emb_56   emb_57   emb_58    emb_59    emb_60    emb_61    emb_62    emb_63   emb_64    emb_65    emb_66    emb_67    emb_68    emb_69    emb_70    emb_71    emb_72    emb_73   emb_74    emb_75    emb_76    emb_77    emb_78    emb_79    emb_80   emb_81    emb_82    emb_83    emb_84   emb_85   emb_86    emb_87    emb_88    emb_89    emb_90    emb_91   emb_92   emb_93    emb_94    emb_95    emb_96    emb_97    emb_98    emb_99  emb_100   emb_101  emb_10

In [4]:
# Split por paciente para garantir que todos os patches de uma imagem fiquem no mesmo conjunto
unique_patients = df["patient_id"].drop_duplicates().to_numpy()
train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

train_mask = df["patient_id"].isin(train_patients)
test_mask = df["patient_id"].isin(test_patients)

X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[test_mask].reset_index(drop=True)
y_train = y.loc[train_mask].reset_index(drop=True)
y_test = y.loc[test_mask].reset_index(drop=True)

print(f"Treino: {len(X_train)} linhas | Teste: {len(X_test)} linhas")
print(f"Pacientes treino: {len(train_patients)} | Pacientes teste: {len(test_patients)}")

Treino: 40188 linhas | Teste: 10244 linhas
Pacientes treino: 204 | Pacientes teste: 52


In [5]:
# Função para avaliar o modelo

def evaluate_regressor(model, X_test, y_test):
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R2: {r2:.6f}")
    return preds

In [6]:
# XGBoost (MultiOutputRegressor)
model_xgb = MultiOutputRegressor(
    XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=1,
        tree_method="hist"
    ),
    n_jobs=6
)
model_xgb.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,"XGBRegressor(...ree=None, ...)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",6
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...), XGBRegressor(...ree=None, ...), ...]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[<U52](133,)","['Age','Age_NA','Shape_irregular',...,'Diagnosis_not_applicable','x_pos', 'y_pos']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,133
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None


In [7]:

print("\n=== XGBoost ===")
preds = evaluate_regressor(model_xgb, X_test, y_test)


=== XGBoost ===
MAE: 0.133939
RMSE: 0.168725
R2: 0.000971


In [8]:
# LightGBM com wrapper para múltiplos alvos
base_lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=100,
    learning_rate=0.08,
    max_depth=-1,
    num_leaves=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,
    verbose=-1,
)
model_lgbm = MultiOutputRegressor(base_lgbm, n_jobs=-1)
model_lgbm.fit(X_train, y_train)


,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,LGBMRegressor... verbose=-1)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",-1
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[LGBMRegressor... verbose=-1), LGBMRegressor... verbose=-1), LGBMRegressor... verbose=-1), LGBMRegressor... verbose=-1), ...]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[<U52](133,)","['Age','Age_NA','Shape_irregular',...,'Diagnosis_not_applicable','x_pos', 'y_pos']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,133
,num_leaves,20
,learning_rate,0.08
,objective,'regression'
,subsample,0.8
,colsample_bytree,0.8


In [9]:

print("\n=== LightGBM ===")
preds_lgbm = evaluate_regressor(model_lgbm, X_test, y_test)


=== LightGBM ===
MAE: 0.133720
RMSE: 0.168420
R2: 0.005487


In [10]:
# CatBoost nativo com múltiplos alvos (teste rápido)
model_cat_base = CatBoostRegressor(
    loss_function="MultiRMSE",
    iterations=150,
    depth=6,
    learning_rate=0.08,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
)
model_cat = MultiOutputRegressor(model_cat_base, n_jobs=-1)

model_cat.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,CatBoostRegre...verbose=False)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",-1
Name,Type,Value
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[CatBoostRegre...verbose=False), CatBoostRegre...verbose=False), CatBoostRegre...verbose=False), CatBoostRegre...verbose=False), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,0


In [11]:
print("\n=== CatBoost ===")
preds_cat = evaluate_regressor(model_cat, X_test, y_test)


=== CatBoost ===
MAE: 0.133176
RMSE: 0.167706
R2: 0.015404


In [12]:
# Comparação final dos modelos
results = {}
for name, model in {
    "XGBoost": model_xgb,
    "LightGBM": model_lgbm,
    "CatBoost": model_cat,
}.items():
    preds = model.predict(X_test)
    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds),
        "R2": r2_score(y_test, preds),
    }

pd.DataFrame(results).T.sort_values("RMSE")

,MAE,RMSE,R2
CatBoost,0.133176,0.167706,0.015404
LightGBM,0.133720,0.168420,0.005487
XGBoost,0.133939,0.168725,0.000971


In [ ]:
# Salvando os modelos
import json
import joblib

joblib.dump(model_xgb, "xgboost_embeddings.joblib")
joblib.dump(model_lgbm, "lightgbm_embeddings.joblib")
joblib.dump(model_cat, "catboost_embeddings.joblib")

print("Modelos salvos em disco:")
print("- xgboost_embeddings.joblib")
print("- lightgbm_embeddings.joblib")
print("- catboost_embeddings.joblib")

# Salvando as previsões do melhor modelo (XGBoost) em formato compatível com o JSON original
# Estrutura: { "case001.png": {"b64": ..., "shape": [...], "dtype": ...}, ... }
# A saída de cada linha será a embedding prevista para o token correspondente.

# Reordenando as saídas no mesmo formato de token por paciente
all_predictions = {}
for patient_id in sorted(df["patient_id"].unique()):
    patient_rows = df[df["patient_id"] == patient_id].copy()
    patient_idx = patient_rows.index
    patient_X = X.loc[patient_idx]
    predicted = model_xgb.predict(patient_X)

    # Cada linha do paciente gera 1 token (CLS + 196 patches), exatamente como o JSON original
    # Aqui preservamos a ordem do dataframe. Cada token recebe 768 dimensões.
    token_list = []
    for token in predicted:
        token_list.append(token.astype(np.float32).tolist())

    # serialização no formato semelhante ao embeddings_per_patch.json
    # mensagem original: [1, 197, 768] para cada imagem
    arr = np.asarray(token_list, dtype=np.float32)
    payload = {
        "b64": __import__('base64').b64encode(arr.tobytes()).decode("ascii"),
        "shape": list(arr.shape),
        "dtype": str(arr.dtype),
    }
    all_predictions[f"case{int(patient_id):03d}.png"] = payload

json_path = "dataset_breast/infered_embeddings.json"
with open(json_path, "w") as f:
    json.dump(all_predictions, f, ensure_ascii=False)

print(f"Previsões salvas em {json_path}")
print(f"Arquivos gerados: {len(all_predictions)} imagens")


Modelos salvos em disco:
- xgboost_embeddings.joblib
- lightgbm_embeddings.joblib
- catboost_embeddings.joblib
